# Text Mining Final Project: Sentiment, Topic Classification, and NERC

This notebook applies three NLP tasks from the Text Mining course to the official final project test sets:

1. Sentiment analysis at sentence level  
2. Topic classification at sentence level  
3. Named Entity Recognition and Classification (NERC) at token level  

External datasets are used for training the sentiment and topic classifiers. The test sets from canvas are used only for final evaluation.

## 1. Setup

We install and import the required libraries for data loading, preprocessing, classification, evaluation, and NERC.  
The main libraries are:

- `datasets` for loading IMDb from Hugging Face
- `kagglehub` for downloading Kaggle datasets
- `scikit-learn` for TF-IDF and Logistic Regression
- `vaderSentiment` for lexicon-based sentiment analysis
- `spaCy` for pre-trained NERC

In [40]:
!pip install datasets kagglehub vaderSentiment spacy seqeval -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [41]:
import os
import pandas as pd
import numpy as np

from datasets import load_dataset
import kagglehub

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import spacy
nlp = spacy.load("en_core_web_sm")

## 2. Data Sources

The project uses two types of data:

### Official test sets
These were provided by the course and are used only for final evaluation:

- `Sentiment-topic-test.tsv`: sentence-level gold labels for sentiment and topic
- `NER-test.tsv`: token-level BIO labels for NERC

### External training data and tools
External datasets are used to train the sentiment and topic classifiers:

- IMDb reviews from Hugging Face (`stanfordnlp/imdb`): https://huggingface.co/datasets/stanfordnlp/imdb
- Restaurant reviews from Kaggle (`ziadmostafa1/restaurant-reviews`): https://www.kaggle.com/datasets/ziadmostafa1/restaurant-reviews
- Amazon Kindle book reviews from Kaggle (`meetnagadia/amazon-kindle-book-review-for-sentiment-analysis`): https://www.kaggle.com/datasets/meetnagadia/amazon-kindle-book-review-for-sentiment-analysis
- spaCy English NER model (`en_core_web_sm`): https://spacy.io/models/en#en_core_web_sm

For NERC, we use spaCy's pre-trained English NER model instead of training a new NER model from scratch.

In [42]:
# Load official test sets provided for the final project
sent_topic_test = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")
ner_test = pd.read_csv("NER-test.tsv", sep="\t")

print("Sentiment-topic test set:")
print(sent_topic_test.head())
print(sent_topic_test.columns)
print(sent_topic_test.shape)

print("\nNER test set:")
print(ner_test.head())
print(ner_test.columns)
print(ner_test.shape)

Sentiment-topic test set:
   sentence id                                               text sentiment  \
0            0  It took eight years for Warner Brothers to rec...  negative   
1            1  All the New York University students love this...  positive   
2            2  This Italian place is really trendy but they h...  negative   
3            3  In conclusion, my review of this book would be...  positive   
4            4  The story of this movie is focused on Carl Bra...   neutral   

        topic  
0       movie  
1  restaurant  
2  restaurant  
3        book  
4       movie  
Index(['sentence id', 'text', 'sentiment', 'topic'], dtype='object')
(10, 4)

NER test set:
   sentence id  token id  token BIO NER tag
0            0         0     It           O
1            0         1   took           O
2            0         2  eight           O
3            0         3  years           O
4            0         4    for           O
Index(['sentence id', 'token id', 'token', 'BIO

In [43]:
imdb = load_dataset("stanfordnlp/imdb")

print(imdb)
print(imdb["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [44]:
imdb_train = pd.DataFrame(imdb["train"])

# IMDb labels: 0 = negative, 1 = positive
imdb_train["sentiment"] = imdb_train["label"].map({0: "negative", 1: "positive"})
imdb_train["topic"] = "movie"

imdb_train = imdb_train[["text", "sentiment", "topic"]]

print(imdb_train.head())
print(imdb_train["sentiment"].value_counts())
print(imdb_train.shape)

                                                text sentiment  topic
0  I rented I AM CURIOUS-YELLOW from my video sto...  negative  movie
1  "I Am Curious: Yellow" is a risible and preten...  negative  movie
2  If only to avoid making this type of film in t...  negative  movie
3  This film was probably inspired by Godard's Ma...  negative  movie
4  Oh, brother...after hearing about this ridicul...  negative  movie
sentiment
negative    12500
positive    12500
Name: count, dtype: int64
(25000, 3)


In [45]:
# Download restaurant reviews dataset from Kaggle
restaurant_path = kagglehub.dataset_download("ziadmostafa1/restaurant-reviews")

print("Restaurant dataset path:", restaurant_path)

# Show files inside the downloaded folder
for root, dirs, files in os.walk(restaurant_path):
    for file in files:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'restaurant-reviews' dataset.
Restaurant dataset path: /kaggle/input/restaurant-reviews
/kaggle/input/restaurant-reviews/Restaurant_Reviews.tsv


In [46]:
# Find CSV or TSV files
restaurant_files = []

for root, dirs, files in os.walk(restaurant_path):
    for file in files:
        if file.endswith(".tsv") or file.endswith(".csv"):
            restaurant_files.append(os.path.join(root, file))

print(restaurant_files)

['/kaggle/input/restaurant-reviews/Restaurant_Reviews.tsv']


In [47]:
# Load restaurant dataset
restaurant_df = pd.read_csv(restaurant_files[0], sep="\t")

print(restaurant_df.head())
print(restaurant_df.columns)
print(restaurant_df.shape)

                                              Review  Liked
0                           Wow... Loved this place.      1
1                                 Crust is not good.      0
2          Not tasty and the texture was just nasty.      0
3  Stopped by during the late May bank holiday of...      1
4  The selection on the menu was great and so wer...      1
Index(['Review', 'Liked'], dtype='object')
(1000, 2)


In [48]:
# Convert restaurant dataset into project format
restaurant_train = restaurant_df.copy()

restaurant_train = restaurant_train.rename(columns={
    "Review": "text",
    "Liked": "label",
    "review": "text",
    "liked": "label"
})

restaurant_train["sentiment"] = restaurant_train["label"].map({
    1: "positive",
    0: "negative"
})

restaurant_train["topic"] = "restaurant"

restaurant_train = restaurant_train[["text", "sentiment", "topic"]]

print(restaurant_train.head())
print(restaurant_train["sentiment"].value_counts())
print(restaurant_train.shape)

                                                text sentiment       topic
0                           Wow... Loved this place.  positive  restaurant
1                                 Crust is not good.  negative  restaurant
2          Not tasty and the texture was just nasty.  negative  restaurant
3  Stopped by during the late May bank holiday of...  positive  restaurant
4  The selection on the menu was great and so wer...  positive  restaurant
sentiment
positive    500
negative    500
Name: count, dtype: int64
(1000, 3)


In [49]:
# Download Amazon Kindle book reviews dataset from Kaggle
# This dataset is used as the book-domain training data
book_path = kagglehub.dataset_download("meetnagadia/amazon-kindle-book-review-for-sentiment-analysis")

print("Book dataset path:", book_path)
# Print all files in the downloaded folder to inspect available CSV files
for root, dirs, files in os.walk(book_path):
    for file in files:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'amazon-kindle-book-review-for-sentiment-analysis' dataset.
Book dataset path: /kaggle/input/amazon-kindle-book-review-for-sentiment-analysis
/kaggle/input/amazon-kindle-book-review-for-sentiment-analysis/all_kindle_review .csv
/kaggle/input/amazon-kindle-book-review-for-sentiment-analysis/preprocessed_kindle_review .csv


In [50]:
# Find all CSV files in the downloaded book dataset folder
book_files = []

for root, dirs, files in os.walk(book_path):
    for file in files:
        if file.endswith(".csv"):
            book_files.append(os.path.join(root, file))

print(book_files)

['/kaggle/input/amazon-kindle-book-review-for-sentiment-analysis/all_kindle_review .csv', '/kaggle/input/amazon-kindle-book-review-for-sentiment-analysis/preprocessed_kindle_review .csv']


In [51]:
# Use the preprocessed Kindle review dataset
book_file = [f for f in book_files if "preprocessed" in f][0]

book_df = pd.read_csv(book_file)

# Convert book dataset into project format
book_train = book_df.copy()

# Keep only rows with review text and rating
book_train = book_train.dropna(subset=["reviewText", "rating"])

# Rename columns
book_train = book_train.rename(columns={
    "reviewText": "text"
})

# Convert rating to numeric
book_train["rating"] = pd.to_numeric(book_train["rating"], errors="coerce")
book_train = book_train.dropna(subset=["rating"])

# Map ratings to sentiment
def rating_to_sentiment(rating):
    if rating >= 4:
        return "positive"
    elif rating <= 2:
        return "negative"
    else:
        return "neutral"

book_train["sentiment"] = book_train["rating"].apply(rating_to_sentiment)

# Add topic label
book_train["topic"] = "book"

# Keep only needed columns
book_train = book_train[["text", "sentiment", "topic"]]

print(book_train.head())
print(book_train["sentiment"].value_counts())
print(book_train["topic"].value_counts())
print(book_train.shape)

                                                text sentiment topic
0  This book was the very first bookmobile book I...  positive  book
1  When I read the description for this book, I c...  negative  book
2  I just had to edit this review. This book is a...  positive  book
3  I don't normally buy 'mystery' novels because ...  positive  book
4  This isn't the kind of book I normally read, a...  positive  book
sentiment
positive    6000
negative    4000
neutral     2000
Name: count, dtype: int64
topic
book    12000
Name: count, dtype: int64
(12000, 3)


## 3. Balanced Training Sets

After loading and cleaning the external datasets, we create two separate balanced training sets.

For topic classification, we use 1000 examples from each domain: movie, restaurant, and book.  
For sentiment classification, we use 1000 examples from each sentiment label: positive, negative, and neutral.

This is done to avoid class imbalance and to make the training setup clearer.

In [52]:
# Create balanced topic training dataset
movie_topic = imdb_train.sample(n=1000, random_state=42)
restaurant_topic = restaurant_train.sample(n=1000, random_state=42)
book_topic = book_train.sample(n=1000, random_state=42)

topic_train = pd.concat(
    [movie_topic, restaurant_topic, book_topic],
    ignore_index=True
)

# Shuffle dataset
topic_train = topic_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(topic_train.head())
print(topic_train["topic"].value_counts())
print(topic_train.shape)

                                                text sentiment       topic
0            The real disappointment was our waiter.  negative  restaurant
1         This place is horrible and way overpriced.  negative  restaurant
2        Overall, I like there food and the service.  positive  restaurant
3  "Night of the Living Homeless" was a fairly st...  positive       movie
4  Life in palm springs is interesting  and the m...  positive        book
topic
restaurant    1000
movie         1000
book          1000
Name: count, dtype: int64
(3000, 3)


For sentiment analysis, positive and negative examples are sampled from all three domains where possible, while neutral examples come from the Kindle book dataset because the IMDb and restaurant datasets only provide binary sentiment labels.

In [53]:
# Create a domain-aware balanced sentiment training dataset

# Positive examples: include all three domains
positive_movie = imdb_train[imdb_train["sentiment"] == "positive"].sample(n=400, random_state=42)
positive_restaurant = restaurant_train[restaurant_train["sentiment"] == "positive"].sample(n=300, random_state=42)
positive_book = book_train[book_train["sentiment"] == "positive"].sample(n=300, random_state=42)

# Negative examples: include all three domains
negative_movie = imdb_train[imdb_train["sentiment"] == "negative"].sample(n=400, random_state=42)
negative_restaurant = restaurant_train[restaurant_train["sentiment"] == "negative"].sample(n=300, random_state=42)
negative_book = book_train[book_train["sentiment"] == "negative"].sample(n=300, random_state=42)

# Neutral examples are only available from the Kindle book dataset
neutral_book = book_train[book_train["sentiment"] == "neutral"].sample(n=1000, random_state=42)

sentiment_train = pd.concat(
    [
        positive_movie, positive_restaurant, positive_book,
        negative_movie, negative_restaurant, negative_book,
        neutral_book
    ],
    ignore_index=True
)

# Shuffle dataset
sentiment_train = sentiment_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(sentiment_train.head())
print("Sentiment distribution:")
print(sentiment_train["sentiment"].value_counts())
print("\nDomain/topic distribution:")
print(sentiment_train["topic"].value_counts())
print(sentiment_train.shape)

                                                text sentiment  topic
0  I couldn't believe a published book could be s...  negative   book
1  Worst movie on earth. I don't even know where ...  negative  movie
2  This was a disappointment.  The author gives t...  negative   book
3  Cusack does his best David Niven in this one, ...  positive  movie
4  I read the book quite a while back, and while ...   neutral   book
Sentiment distribution:
sentiment
negative    1000
positive    1000
neutral     1000
Name: count, dtype: int64

Domain/topic distribution:
topic
book          1600
movie          800
restaurant     600
Name: count, dtype: int64
(3000, 3)


## 4. Sentiment Analysis

We compare two sentiment systems on the official sentence-level test set.

The first system is VADER, a lexicon-based method that uses sentiment words and rules.  
The second system is TF-IDF + Logistic Regression, a supervised machine learning classifier trained on the balanced sentiment training set.

In [54]:
# Sentiment System 1: VADER baseline
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

sent_topic_test["vader_sentiment"] = sent_topic_test["text"].apply(vader_sentiment)

print(sent_topic_test[["sentence id", "text", "sentiment", "vader_sentiment"]])

print("\nVADER classification report:")
print(classification_report(
    sent_topic_test["sentiment"],
    sent_topic_test["vader_sentiment"],
    labels=["positive", "negative", "neutral"]
))

   sentence id                                               text sentiment  \
0            0  It took eight years for Warner Brothers to rec...  negative   
1            1  All the New York University students love this...  positive   
2            2  This Italian place is really trendy but they h...  negative   
3            3  In conclusion, my review of this book would be...  positive   
4            4  The story of this movie is focused on Carl Bra...   neutral   
5            5  Chris O'Donnell stated that while filming for ...   neutral   
6            6  My husband and I moved to Amsterdam 6 years ag...  positive   
7            7  Dame Maggie Smith performed her role excellent...  positive   
8            8  The new movie by Mr. Kruno was shot in New Yor...   neutral   
9            9  I always have loved English novels, but I just...  negative   

  vader_sentiment  
0        negative  
1        positive  
2        positive  
3        positive  
4        positive  
5        p

In [55]:
# Sentiment System 2: TF-IDF + Logistic Regression

sent_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_sent_train = sent_vectorizer.fit_transform(sentiment_train["text"])
y_sent_train = sentiment_train["sentiment"]

sent_clf = LogisticRegression(max_iter=1000, random_state=42)
sent_clf.fit(X_sent_train, y_sent_train)

X_sent_test = sent_vectorizer.transform(sent_topic_test["text"])
sent_topic_test["tfidf_lr_sentiment"] = sent_clf.predict(X_sent_test)

print(sent_topic_test[["sentence id", "text", "sentiment", "vader_sentiment", "tfidf_lr_sentiment"]])

print("\nTF-IDF + Logistic Regression sentiment classification report:")
print(classification_report(
    sent_topic_test["sentiment"],
    sent_topic_test["tfidf_lr_sentiment"],
    labels=["positive", "negative", "neutral"]
))

   sentence id                                               text sentiment  \
0            0  It took eight years for Warner Brothers to rec...  negative   
1            1  All the New York University students love this...  positive   
2            2  This Italian place is really trendy but they h...  negative   
3            3  In conclusion, my review of this book would be...  positive   
4            4  The story of this movie is focused on Carl Bra...   neutral   
5            5  Chris O'Donnell stated that while filming for ...   neutral   
6            6  My husband and I moved to Amsterdam 6 years ag...  positive   
7            7  Dame Maggie Smith performed her role excellent...  positive   
8            8  The new movie by Mr. Kruno was shot in New Yor...   neutral   
9            9  I always have loved English novels, but I just...  negative   

  vader_sentiment tfidf_lr_sentiment  
0        negative           negative  
1        positive           positive  
2        posi

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 5. Topic Classification

Topic classification is performed using TF-IDF + Logistic Regression.

The model is trained on the balanced topic training set with three labels: movie, restaurant, and book. It is then evaluated on the official sentence-level test set.

In [56]:
# Topic classification: TF-IDF + Logistic Regression

topic_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

X_topic_train = topic_vectorizer.fit_transform(topic_train["text"])
y_topic_train = topic_train["topic"]

topic_clf = LogisticRegression(max_iter=1000, random_state=42)
topic_clf.fit(X_topic_train, y_topic_train)

X_topic_test = topic_vectorizer.transform(sent_topic_test["text"])
sent_topic_test["tfidf_lr_topic"] = topic_clf.predict(X_topic_test)

print(sent_topic_test[["sentence id", "text", "topic", "tfidf_lr_topic"]])

print("\nTF-IDF + Logistic Regression topic classification report:")
print(classification_report(
    sent_topic_test["topic"],
    sent_topic_test["tfidf_lr_topic"],
    labels=["movie", "restaurant", "book"]
))

   sentence id                                               text       topic  \
0            0  It took eight years for Warner Brothers to rec...       movie   
1            1  All the New York University students love this...  restaurant   
2            2  This Italian place is really trendy but they h...  restaurant   
3            3  In conclusion, my review of this book would be...        book   
4            4  The story of this movie is focused on Carl Bra...       movie   
5            5  Chris O'Donnell stated that while filming for ...       movie   
6            6  My husband and I moved to Amsterdam 6 years ag...  restaurant   
7            7  Dame Maggie Smith performed her role excellent...       movie   
8            8  The new movie by Mr. Kruno was shot in New Yor...       movie   
9            9  I always have loved English novels, but I just...        book   

  tfidf_lr_topic  
0          movie  
1     restaurant  
2     restaurant  
3           book  
4          mo

## 6. Named Entity Recognition and Classification

For NERC, we use spaCy's pre-trained English NER model.

The official NER test set is token-level and uses BIO labels. Therefore, we first reconstruct the tokens into sentences, run spaCy on the reconstructed sentences, convert spaCy's entity spans back into BIO tags, and compare them with the gold labels.

In [57]:
# Reconstruct full sentences from token-level NER test set

ner_sentences = (
    ner_test
    .groupby("sentence id")["token"]
    .apply(list)
    .reset_index()
)

ner_sentences["text"] = ner_sentences["token"].apply(lambda tokens: " ".join(tokens))

print(ner_sentences.head())
print("Number of sentences:", len(ner_sentences))

   sentence id                                              token  \
0            0  [It, took, eight, years, for, Warner, Brothers...   
1            1  [All, the, New, York, University, students, lo...   
2            2  [This, Italian, place, is, really, trendy, but...   
3            3  [In, conclusion, ,, my, review, of, this, book...   
4            4  [The, story, of, this, movie, is, focused, on,...   

                                                text  
0  It took eight years for Warner Brothers to rec...  
1  All the New York University students love this...  
2  This Italian place is really trendy but they h...  
3  In conclusion , my review of this book would b...  
4  The story of this movie is focused on Carl Bra...  
Number of sentences: 10


In [58]:
print("Gold BIO label distribution:")
print(ner_test["BIO NER tag"].value_counts())

Gold BIO label distribution:
BIO NER tag
O         183
I-PER       8
B-PER       6
B-ORG       4
B-LOC       4
B-MISC      3
I-ORG       3
I-LOC       2
I-MISC      1
Name: count, dtype: int64


In [59]:
# Map spaCy entity labels to the course labels
def map_spacy_label(label):
    if label == "PERSON":
        return "PER"
    elif label == "ORG":
        return "ORG"
    elif label in ["GPE", "LOC", "FAC"]:
        return "LOC"
    elif label in ["NORP", "EVENT", "WORK_OF_ART", "LAW", "LANGUAGE", "PRODUCT"]:
        return "MISC"
    else:
        return None

In [60]:
def spacy_bio_tags(tokens):
    text = " ".join(tokens)
    doc = nlp(text)

    bio_tags = ["O"] * len(tokens)

    # Create character offsets for each original token
    offsets = []
    current = 0

    for token in tokens:
        start = current
        end = current + len(token)
        offsets.append((start, end))
        current = end + 1  # space after token

    # Assign BIO tags where spaCy entity spans overlap with tokens
    for ent in doc.ents:
        mapped_label = map_spacy_label(ent.label_)

        if mapped_label is None:
            continue

        ent_start = ent.start_char
        ent_end = ent.end_char

        inside_entity = []

        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_start < ent_end and tok_end > ent_start:
                inside_entity.append(i)

        for j, token_index in enumerate(inside_entity):
            prefix = "B" if j == 0 else "I"
            bio_tags[token_index] = f"{prefix}-{mapped_label}"

    return bio_tags

In [61]:
predicted_rows = []

for _, row in ner_sentences.iterrows():
    sent_id = row["sentence id"]
    tokens = row["token"]
    pred_tags = spacy_bio_tags(tokens)

    for token_id, token, pred_tag in zip(range(len(tokens)), tokens, pred_tags):
        predicted_rows.append({
            "sentence id": sent_id,
            "token id": token_id,
            "token": token,
            "spacy_bio_pred": pred_tag
        })

ner_pred = pd.DataFrame(predicted_rows)

print(ner_pred.head(20))
print("\nPredicted BIO label distribution:")
print(ner_pred["spacy_bio_pred"].value_counts())
print(ner_pred.shape)

    sentence id  token id     token spacy_bio_pred
0             0         0        It              O
1             0         1      took              O
2             0         2     eight              O
3             0         3     years              O
4             0         4       for              O
5             0         5    Warner          B-ORG
6             0         6  Brothers          I-ORG
7             0         7        to              O
8             0         8   recover              O
9             0         9      from              O
10            0        10       the              O
11            0        11  disaster              O
12            0        12      that              O
13            0        13       was              O
14            0        14      this              O
15            0        15     movie              O
16            0        16         .              O
17            1         0       All              O
18            1         1      

In [62]:
ner_eval = ner_test.merge(
    ner_pred,
    on=["sentence id", "token id", "token"],
    how="left"
)

print(ner_eval.head(30))
print(ner_eval.shape)
print("Missing predictions:", ner_eval["spacy_bio_pred"].isna().sum())

    sentence id  token id       token BIO NER tag spacy_bio_pred
0             0         0          It           O              O
1             0         1        took           O              O
2             0         2       eight           O              O
3             0         3       years           O              O
4             0         4         for           O              O
5             0         5      Warner       B-ORG          B-ORG
6             0         6    Brothers       I-ORG          I-ORG
7             0         7          to           O              O
8             0         8     recover           O              O
9             0         9        from           O              O
10            0        10         the           O              O
11            0        11    disaster           O              O
12            0        12        that           O              O
13            0        13         was           O              O
14            0        14

In [63]:
ner_labels = [
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC",
    "O"
]

print("spaCy NERC token-level classification report:")
print(classification_report(
    ner_eval["BIO NER tag"],
    ner_eval["spacy_bio_pred"],
    labels=ner_labels,
    zero_division=0
))

spaCy NERC token-level classification report:
              precision    recall  f1-score   support

       B-PER       0.67      0.67      0.67         6
       I-PER       1.00      0.75      0.86         8
       B-ORG       0.67      0.50      0.57         4
       I-ORG       0.75      1.00      0.86         3
       B-LOC       1.00      1.00      1.00         4
       I-LOC       1.00      1.00      1.00         2
      B-MISC       0.75      1.00      0.86         3
      I-MISC       1.00      1.00      1.00         1
           O       0.99      0.99      0.99       183

    accuracy                           0.97       214
   macro avg       0.87      0.88      0.87       214
weighted avg       0.97      0.97      0.97       214



In [64]:
ner_eval.to_csv("ner_spacy_predictions.csv", index=False)

print("Saved ner_spacy_predictions.csv")

Saved ner_spacy_predictions.csv


In [65]:
# Save all final prediction/evaluation files
sent_topic_test.to_csv("sentiment_topic_predictions.csv", index=False)
ner_eval.to_csv("ner_spacy_predictions.csv", index=False)

topic_train.to_csv("topic_training_sample.csv", index=False)
sentiment_train.to_csv("sentiment_training_sample.csv", index=False)

print("Saved files:")
print("- sentiment_topic_predictions.csv")
print("- ner_spacy_predictions.csv")
print("- topic_training_sample.csv")
print("- sentiment_training_sample.csv")

Saved files:
- sentiment_topic_predictions.csv
- ner_spacy_predictions.csv
- topic_training_sample.csv
- sentiment_training_sample.csv


## 7. Results, Dataset Statistics, and Error Analysis

This section collects the outputs needed for the final poster: dataset statistics, result summaries, and qualitative error examples for sentiment, topic classification, and NERC.

In [66]:
print("=== OFFICIAL TEST SETS ===")

print("\nSentiment-topic test set:")
print("Number of sentences:", len(sent_topic_test))
print("\nGold sentiment distribution:")
print(sent_topic_test["sentiment"].value_counts())
print("\nGold topic distribution:")
print(sent_topic_test["topic"].value_counts())

print("\nNER test set:")
print("Number of tokens:", len(ner_test))
print("Number of sentences:", ner_test["sentence id"].nunique())
print("\nGold BIO label distribution:")
print(ner_test["BIO NER tag"].value_counts())


print("\n=== TRAINING DATA ===")

print("\nSentiment training set:")
print("Number of examples:", len(sentiment_train))
print(sentiment_train["sentiment"].value_counts())
print(sentiment_train["topic"].value_counts())

print("\nTopic training set:")
print("Number of examples:", len(topic_train))
print(topic_train["topic"].value_counts())

=== OFFICIAL TEST SETS ===

Sentiment-topic test set:
Number of sentences: 10

Gold sentiment distribution:
sentiment
positive    4
negative    3
neutral     3
Name: count, dtype: int64

Gold topic distribution:
topic
movie         5
restaurant    3
book          2
Name: count, dtype: int64

NER test set:
Number of tokens: 214
Number of sentences: 10

Gold BIO label distribution:
BIO NER tag
O         183
I-PER       8
B-PER       6
B-ORG       4
B-LOC       4
B-MISC      3
I-ORG       3
I-LOC       2
I-MISC      1
Name: count, dtype: int64

=== TRAINING DATA ===

Sentiment training set:
Number of examples: 3000
sentiment
negative    1000
positive    1000
neutral     1000
Name: count, dtype: int64
topic
book          1600
movie          800
restaurant     600
Name: count, dtype: int64

Topic training set:
Number of examples: 3000
topic
restaurant    1000
movie         1000
book          1000
Name: count, dtype: int64


In [71]:
results_summary = pd.DataFrame({
    "Task": [
        "Sentiment analysis",
        "Sentiment analysis",
        "Topic classification",
        "NERC"
    ],
    "System": [
        "VADER",
        "TF-IDF + Logistic Regression",
        "TF-IDF + Logistic Regression",
        "spaCy pre-trained NER"
    ],
    "Official test set": [
        "Sentiment-topic-test.tsv",
        "Sentiment-topic-test.tsv",
        "Sentiment-topic-test.tsv",
        "NER-test.tsv"
    ],
    "Main result": [
        "Accuracy = 0.60, Macro F1 = 0.56",
        "Accuracy = 0.50, Macro F1 = 0.39",
        "Accuracy = 0.90, Macro F1 = 0.92",
        "Accuracy = 0.97, Macro F1 = 0.87"
    ]
})

print(results_summary)
results_summary.to_csv("results_summary.csv", index=False)

                   Task                        System  \
0    Sentiment analysis                         VADER   
1    Sentiment analysis  TF-IDF + Logistic Regression   
2  Topic classification  TF-IDF + Logistic Regression   
3                  NERC         spaCy pre-trained NER   

          Official test set                       Main result  
0  Sentiment-topic-test.tsv  Accuracy = 0.60, Macro F1 = 0.56  
1  Sentiment-topic-test.tsv  Accuracy = 0.50, Macro F1 = 0.39  
2  Sentiment-topic-test.tsv  Accuracy = 0.90, Macro F1 = 0.92  
3              NER-test.tsv  Accuracy = 0.97, Macro F1 = 0.87  


In [72]:
sentiment_errors = sent_topic_test[
    (sent_topic_test["sentiment"] != sent_topic_test["vader_sentiment"]) |
    (sent_topic_test["sentiment"] != sent_topic_test["tfidf_lr_sentiment"])
][["sentence id", "text", "sentiment", "vader_sentiment", "tfidf_lr_sentiment"]]

print(sentiment_errors)
sentiment_errors.to_csv("sentiment_error_examples.csv", index=False)

   sentence id                                               text sentiment  \
2            2  This Italian place is really trendy but they h...  negative   
3            3  In conclusion, my review of this book would be...  positive   
4            4  The story of this movie is focused on Carl Bra...   neutral   
5            5  Chris O'Donnell stated that while filming for ...   neutral   
8            8  The new movie by Mr. Kruno was shot in New Yor...   neutral   
9            9  I always have loved English novels, but I just...  negative   

  vader_sentiment tfidf_lr_sentiment  
2        positive           positive  
3        positive           negative  
4        positive           positive  
5        positive           negative  
8         neutral           positive  
9        positive           negative  


In [73]:
topic_errors = sent_topic_test[
    sent_topic_test["topic"] != sent_topic_test["tfidf_lr_topic"]
][["sentence id", "text", "topic", "tfidf_lr_topic"]]

print(topic_errors)
topic_errors.to_csv("topic_error_examples.csv", index=False)

   sentence id                                               text  topic  \
7            7  Dame Maggie Smith performed her role excellent...  movie   

  tfidf_lr_topic  
7     restaurant  


In [74]:
ner_errors = ner_eval[
    ner_eval["BIO NER tag"] != ner_eval["spacy_bio_pred"]
][["sentence id", "token id", "token", "BIO NER tag", "spacy_bio_pred"]]

print(ner_errors)
ner_errors.to_csv("ner_error_examples.csv", index=False)

     sentence id  token id      token BIO NER tag spacy_bio_pred
18             1         1        the           O          B-ORG
19             1         2        New       B-ORG          I-ORG
151            6        20  Blauwbrug       B-ORG         B-MISC
160            7         0       Dame       B-PER              O
161            7         1     Maggie       I-PER          B-PER
180            8         4        Mr.       B-PER              O
181            8         5      Kruno       I-PER          B-PER


## 8. Final Outputs

This notebook produced the following files for the final poster:

- sentiment_topic_predictions.csv
- ner_spacy_predictions.csv
- topic_training_sample.csv
- sentiment_training_sample.csv
- results_summary.csv
- sentiment_error_examples.csv
- topic_error_examples.csv
- ner_error_examples.csv

The official test sets were used only for final evaluation. External datasets were used for training the sentiment and topic classifiers. For NERC, a pre-trained spaCy model was evaluated on the official BIO-tagged test set.